# عرض توضيحي للمشابك العصبية المخصصة لـ SRA (قاعدة بيانات المتجهات والآلة الحاسبة).

In [ ]:
import sys
import os

# Setup for running on Google Colab
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn.functional as F
from sra_reference import SRAModel, VectorDBSynapse, CalculatorSynapse

## 1. تهيئة نموذج SRA

In [ ]:
# Initialize the SRA model (with 4 ordinary neural synapses)
vocab_size = 100
dim = 32
layers = 2
num_synapses = 4
k = 1
syn_hidden = 64

model = SRAModel(vocab_size, dim, layers, num_synapses, k, syn_hidden)
print(f"Initial number of synapses: {model.blocks[0].router.num_synapses}")

## 2. إضافة المشابك العصبية المخصصة بسرعة التبديل

In [ ]:
# Add a VectorDBSynapse
def create_vector_db():
    return VectorDBSynapse(dim)

def create_vector_db_emb():
    # Assign an embedding for a specific domain (random)
    return torch.randn(dim)

model.add_custom_synapse(create_vector_db, create_vector_db_emb)
print(f"Number of synapses after adding VectorDBSynapse: {model.blocks[0].router.num_synapses}")

# Add a CalculatorSynapse
def create_calculator():
    return CalculatorSynapse(dim)

def create_calc_emb():
    return torch.randn(dim)

model.add_custom_synapse(create_calculator, create_calc_emb)
print(f"Number of synapses after adding CalculatorSynapse: {model.blocks[0].router.num_synapses}")

## 3. تسجيل واختبار المعرفة في VectorDB Synapse

In [ ]:
# Add knowledge to the VectorDB synapse
# Retrieve the VectorDB synapse (index 4) of the first block
db_synapse = model.blocks[0].synapses[4]

# Register a feature vector (mock) corresponding to "What is the capital of Paris?" and its answer (fact vector)
fact_key = torch.randn(dim)
fact_value = torch.ones(dim) * 99  # A distinctive value

db_synapse.add_knowledge(fact_key, fact_value)
print("Registered the fact data into the VectorDB.")

# Input test: when we pass a feature exactly matching the fact key
x_in = fact_key.unsqueeze(0).unsqueeze(0)  # (1, 1, D)
out = db_synapse(x_in)
print(f"VectorDB output: {out[0, 0, :5]} ... (expected: 99)")

## 4. التحقق من التوجيه

In [ ]:
# Routing test through the whole model
x = torch.randint(0, vocab_size, (1, 10))
y_in = torch.randint(0, vocab_size, (1, 5))

# Run inference
out, router_logits, syn_outputs = model(x, y_in)

# Inspect the router logits (which synapse each token was routed to)
print(f"Router Logits (Block 0): shape {router_logits[0].shape}")
probs = F.softmax(router_logits[0], dim=-1)
print(f"Routing probability to each synapse (first token): \n{probs[0, 0, :]}")